In [1]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

text = "Prior authorization is required for MRI procedures."

tokens = encoding.encode(text)

print("Original text:", text)
print("Word count:", len(text.split()))
print("Token count:", len(tokens))
print("Token IDs:", tokens)

Original text: Prior authorization is required for MRI procedures.
Word count: 7
Token count: 8
Token IDs: [50571, 24645, 374, 2631, 369, 52460, 16346, 13]


In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model = os.getenv("AZURE_OPENAI_MODEL")

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

print("Azure model client ready")

Azure model client ready


In [2]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} -> {repr(token_text)}")

50571 -> 'Prior'
24645 -> ' authorization'
374 -> ' is'
2631 -> ' required'
369 -> ' for'
52460 -> ' MRI'
16346 -> ' procedures'
13 -> '.'


In [3]:
samples = [
    "Prior authorization is required for MRI procedures.",
    "PA is required for MRI.",
    "The member's deductible is $1,500.",
    "AuthorizationRequired=True",
    "पूर्व प्राधिकरण आवश्यक है"
]

for text in samples:
    token_ids = encoding.encode(text)

    print("\nTEXT:", text)
    print("Words :", len(text.split()))
    print("Tokens:", len(token_ids))


TEXT: Prior authorization is required for MRI procedures.
Words : 7
Tokens: 8

TEXT: PA is required for MRI.
Words : 5
Tokens: 6

TEXT: The member's deductible is $1,500.
Words : 5
Tokens: 10

TEXT: AuthorizationRequired=True
Words : 1
Tokens: 3

TEXT: पूर्व प्राधिकरण आवश्यक है
Words : 4
Tokens: 27


In [4]:
samples = [
    "Prior authorization is required for MRI procedures.",
    "PA is required for MRI.",
    "The member's deductible is $1,500.",
    "AuthorizationRequired=True",
    "पूर्व प्राधिकरण आवश्यक है"
]

for text in samples:
    words = len(text.split())
    tokens = len(encoding.encode(text))
    ratio = tokens / words if words > 0 else 0

    print(f"\n{text}")
    print(f"Words            : {words}")
    print(f"Tokens           : {tokens}")
    print(f"Tokens per word  : {ratio:.2f}")


Prior authorization is required for MRI procedures.
Words            : 7
Tokens           : 8
Tokens per word  : 1.14

PA is required for MRI.
Words            : 5
Tokens           : 6
Tokens per word  : 1.20

The member's deductible is $1,500.
Words            : 5
Tokens           : 10
Tokens per word  : 2.00

AuthorizationRequired=True
Words            : 1
Tokens           : 3
Tokens per word  : 3.00

पूर्व प्राधिकरण आवश्यक है
Words            : 4
Tokens           : 27
Tokens per word  : 6.75


In [7]:
prompt_text = "Explain prior authorization in healthcare in two sentences."

local_token_count = len(encoding.encode(prompt_text))

api_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": prompt_text
        }
    ]
)

print("Local text token estimate :", local_token_count)
print("Azure prompt tokens       :", api_response.usage.prompt_tokens)

Local text token estimate : 10
Azure prompt tokens       : 16


In [8]:
short_context = """
Policy: MRI procedures require prior authorization.
"""

long_context = """
Policy: MRI procedures require prior authorization.
CT scans require prior authorization only for outpatient procedures.
Emergency room imaging does not require prior authorization.
Physical therapy requires authorization after 10 visits.
Specialist consultations do not require prior authorization.
"""

question = "Does an MRI require prior authorization?"

In [9]:
short_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{short_context}

Question:
{question}
"""
        }
    ]
)

print(short_response.choices[0].message.content)
print("\nPrompt tokens:", short_response.usage.prompt_tokens)

Yes, an MRI requires prior authorization according to the policy.

Prompt tokens: 39


In [10]:
long_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{long_context}

Question:
{question}
"""
        }
    ]
)

print(long_response.choices[0].message.content)
print("\nPrompt tokens:", long_response.usage.prompt_tokens)

Yes, an MRI procedure requires prior authorization.

Prompt tokens: 76


In [11]:
short_tokens = short_response.usage.prompt_tokens
long_tokens = long_response.usage.prompt_tokens

increase = long_tokens - short_tokens
increase_pct = (increase / short_tokens) * 100

print("Short-context tokens :", short_tokens)
print("Long-context tokens  :", long_tokens)
print("Additional tokens    :", increase)
print(f"Increase             : {increase_pct:.1f}%")

Short-context tokens : 39
Long-context tokens  : 76
Additional tokens    : 37
Increase             : 94.9%


In [12]:
short_tokens = short_response.usage.prompt_tokens
long_tokens = long_response.usage.prompt_tokens

increase = long_tokens - short_tokens
increase_pct = (increase / short_tokens) * 100

print("Short-context tokens :", short_tokens)
print("Long-context tokens  :", long_tokens)
print("Additional tokens    :", increase)
print(f"Increase             : {increase_pct:.1f}%")

Short-context tokens : 39
Long-context tokens  : 76
Additional tokens    : 37
Increase             : 94.9%


In [13]:
noisy_context = """
Member ID: M102938
Plan Type: PPO
Primary Care Copay: $25
Specialist Copay: $50
Emergency Room Copay: $250
Dental coverage is not included.
Vision coverage is included once every 24 months.
Physical therapy requires authorization after 10 visits.
MRI procedures require prior authorization.
Member mailing address was updated last month.
Claims are processed within standard turnaround time.
"""

question = "Does an MRI require prior authorization?"

In [14]:
noisy_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{noisy_context}

Question:
{question}
"""
        }
    ]
)

print(noisy_response.choices[0].message.content)
print("\nPrompt tokens:", noisy_response.usage.prompt_tokens)

Yes, an MRI procedure requires prior authorization according to the provided policy context.

Prompt tokens: 114


In [15]:
print("Short context tokens :", short_response.usage.prompt_tokens)
print("Long context tokens  :", long_response.usage.prompt_tokens)
print("Noisy context tokens :", noisy_response.usage.prompt_tokens)

Short context tokens : 39
Long context tokens  : 76
Noisy context tokens : 114


In [17]:
import pandas as pd

context_comparison = pd.DataFrame([
    {
        "Scenario": "Short context",
        "Prompt Tokens": short_response.usage.prompt_tokens,
        "Answer Quality": "Correct",
        "Context Relevance": "High"
    },
    {
        "Scenario": "Long context",
        "Prompt Tokens": long_response.usage.prompt_tokens,
        "Answer Quality": "Correct",
        "Context Relevance": "Medium"
    },
    {
        "Scenario": "Noisy context",
        "Prompt Tokens": noisy_response.usage.prompt_tokens,
        "Answer Quality": "Correct",
        "Context Relevance": "Low"
    }
])

context_comparison

,Scenario,Prompt Tokens,Answer Quality,Context Relevance
0,Short context,39,Correct,High
1,Long context,76,Correct,Medium
2,Noisy context,114,Correct,Low


In [18]:
embedding_samples = [
    "Prior authorization is required for MRI procedures.",
    "MRI scans need approval from the health insurance payer.",
    "The member updated their mailing address.",
    "The deductible must be paid before the health plan starts sharing costs."
]

for i, text in enumerate(embedding_samples, start=1):
    print(f"{i}. {text}")

1. Prior authorization is required for MRI procedures.
2. MRI scans need approval from the health insurance payer.
3. The member updated their mailing address.
4. The deductible must be paid before the health plan starts sharing costs.


In [20]:
from dotenv import load_dotenv
import os

load_dotenv("../.env", override=True)

embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

print("Embedding model configured:", bool(embedding_model))
print("Embedding deployment       :", embedding_model)

Embedding model configured: True
Embedding deployment       : text-embedding-3-small


In [21]:
embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

print("Embedding model configured:", bool(embedding_model))
print("Embedding deployment       :", embedding_model)

Embedding model configured: True
Embedding deployment       : text-embedding-3-small


In [22]:
embedding_response = client.embeddings.create(
    model=embedding_model,
    input=embedding_samples
)

embeddings = [item.embedding for item in embedding_response.data]

print("Number of embeddings:", len(embeddings))
print("Embedding dimension :", len(embeddings[0]))

Number of embeddings: 4
Embedding dimension : 1536


In [23]:
import numpy as np

def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    return np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )

for i in range(len(embedding_samples)):
    for j in range(i + 1, len(embedding_samples)):
        score = cosine_similarity(
            embeddings[i],
            embeddings[j]
        )

        print(f"{i+1} vs {j+1}: {score:.4f}")

1 vs 2: 0.6701
1 vs 3: 0.0764
1 vs 4: 0.3586
2 vs 3: 0.1100
2 vs 4: 0.4465
3 vs 4: 0.0513


In [24]:
import pandas as pd
import numpy as np

similarity_matrix = np.zeros((len(embedding_samples), len(embedding_samples)))

for i in range(len(embedding_samples)):
    for j in range(len(embedding_samples)):
        similarity_matrix[i][j] = cosine_similarity(
            embeddings[i],
            embeddings[j]
        )

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=[f"Text {i+1}" for i in range(len(embedding_samples))],
    columns=[f"Text {i+1}" for i in range(len(embedding_samples))]
)

similarity_df.round(3)

,Text 1,Text 2,Text 3,Text 4
Text 1,1.000,0.670,0.076,0.359
Text 2,0.670,1.000,0.110,0.447
Text 3,0.076,0.110,1.000,0.051
Text 4,0.359,0.447,0.051,1.000


In [25]:
reasoning_case = """
A member has already completed 8 physical therapy visits.
The health plan policy allows 10 visits without prior authorization.
Prior authorization is required starting from the 11th visit.

The provider is requesting the member's 9th physical therapy visit.
"""

reasoning_question = """
Does this visit require prior authorization?
Give only the final decision and one-line justification.
"""

print(reasoning_case)
print(reasoning_question)


A member has already completed 8 physical therapy visits.
The health plan policy allows 10 visits without prior authorization.
Prior authorization is required starting from the 11th visit.

The provider is requesting the member's 9th physical therapy visit.


Does this visit require prior authorization?
Give only the final decision and one-line justification.



In [26]:
direct_reasoning_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": f"""
{reasoning_case}

{reasoning_question}
"""
        }
    ]
)

print(direct_reasoning_response.choices[0].message.content)

No, prior authorization is not required because the 9th visit is within the allowed 10 visits without prior authorization.


In [27]:
structured_reasoning_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer policy reasoning assistant.

Evaluate the case using this structure:
1. Policy rule
2. Case fact
3. Decision

Keep the answer concise.
"""
        },
        {
            "role": "user",
            "content": f"""
{reasoning_case}

Question:
Does this visit require prior authorization?
"""
        }
    ]
)

print(structured_reasoning_response.choices[0].message.content)

1. Policy rule: Up to 10 physical therapy visits are allowed without prior authorization; prior authorization is required starting from the 11th visit.  
2. Case fact: The member has completed 8 visits; the provider is requesting the 9th visit.  
3. Decision: The 9th visit does not require prior authorization and can be approved without it.


In [28]:
structured_reasoning_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer policy reasoning assistant.

Evaluate the case using this structure:
1. Policy rule
2. Case fact
3. Decision

Keep the answer concise.
"""
        },
        {
            "role": "user",
            "content": f"""
{reasoning_case}

Question:
Does this visit require prior authorization?
"""
        }
    ]
)

print(structured_reasoning_response.choices[0].message.content)

1. Policy rule: Prior authorization is required starting from the 11th physical therapy visit; visits 1 through 10 do not require prior authorization.

2. Case fact: The member has completed 8 visits and is requesting the 9th visit.

3. Decision: The 9th visit does not require prior authorization.


In [29]:
ambiguous_case = """
A member has completed 10 physical therapy visits.

The policy states:
"Prior authorization may be required after the initial covered visits,
depending on the member's plan and clinical review requirements."

The provider is requesting the 11th visit.
"""

ambiguous_question = """
Does the 11th visit require prior authorization?
"""

print(ambiguous_case)
print(ambiguous_question)


A member has completed 10 physical therapy visits.

The policy states:
"Prior authorization may be required after the initial covered visits,
depending on the member's plan and clinical review requirements."

The provider is requesting the 11th visit.


Does the 11th visit require prior authorization?



In [30]:
ambiguous_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": f"""
{ambiguous_case}

{ambiguous_question}
Give a clear decision and a short justification.
"""
        }
    ]
)

print(ambiguous_response.choices[0].message.content)

**Decision:** Yes, the 11th visit requires prior authorization.

**Justification:** According to the policy, prior authorization may be required after the initial covered visits based on the member's plan and clinical review requirements. Since the member has completed the initial 10 visits, the 11th visit likely falls under the visits that need prior authorization before proceeding.


In [31]:
guardrailed_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer policy assistant.

Rules:
1. Answer only from the provided policy.
2. Do not infer missing policy conditions.
3. If the available information is insufficient for a definitive decision,
   clearly state "Insufficient information".
4. Explain what additional information is required.
"""
        },
        {
            "role": "user",
            "content": f"""
{ambiguous_case}

Question:
Does the 11th visit require prior authorization?
"""
        }
    ]
)

print(guardrailed_response.choices[0].message.content)

Insufficient information.

Additional information required:
- The specific member's plan details regarding prior authorization requirements for physical therapy visits beyond the initial covered visits.
- Whether the clinical review for this member's plan necessitates prior authorization after the initial visits.


In [32]:
print("WITHOUT GUARDRAIL")
print("-" * 50)
print(ambiguous_response.choices[0].message.content)

print("\nWITH GUARDRAIL")
print("-" * 50)
print(guardrailed_response.choices[0].message.content)

WITHOUT GUARDRAIL
--------------------------------------------------
**Decision:** Yes, the 11th visit requires prior authorization.

**Justification:** According to the policy, prior authorization may be required after the initial covered visits based on the member's plan and clinical review requirements. Since the member has completed the initial 10 visits, the 11th visit likely falls under the visits that need prior authorization before proceeding.

WITH GUARDRAIL
--------------------------------------------------
Insufficient information.

Additional information required:
- The specific member's plan details regarding prior authorization requirements for physical therapy visits beyond the initial covered visits.
- Whether the clinical review for this member's plan necessitates prior authorization after the initial visits.


In [33]:
hallucination_question = """
According to the ZS Platinum Plus Health Plan 2026,
what is the maximum number of chiropractic visits allowed per year?
"""

print(hallucination_question)


According to the ZS Platinum Plus Health Plan 2026,
what is the maximum number of chiropractic visits allowed per year?



In [34]:
hallucination_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": hallucination_question
        }
    ]
)

print(hallucination_response.choices[0].message.content)

According to the ZS Platinum Plus Health Plan 2026, the maximum number of chiropractic visits allowed per year is 20 visits.


In [35]:
hallucination_guardrail_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer policy assistant.

Rules:
1. Answer only when the required information is present in the provided context.
2. Never invent plan benefits, limits, policies, or coverage rules.
3. If no supporting policy context is provided, respond:
   "I cannot determine this from the available information."
"""
        },
        {
            "role": "user",
            "content": hallucination_question
        }
    ]
)

print(hallucination_guardrail_response.choices[0].message.content)

I cannot determine this from the available information.


In [36]:
print("WITHOUT EVIDENCE GUARDRAIL")
print("-" * 50)
print(hallucination_response.choices[0].message.content)

print("\nWITH EVIDENCE GUARDRAIL")
print("-" * 50)
print(hallucination_guardrail_response.choices[0].message.content)

WITHOUT EVIDENCE GUARDRAIL
--------------------------------------------------
According to the ZS Platinum Plus Health Plan 2026, the maximum number of chiropractic visits allowed per year is 20 visits.

WITH EVIDENCE GUARDRAIL
--------------------------------------------------
I cannot determine this from the available information.


In [37]:
load_dotenv("../.env", override=True)

model_primary = os.getenv("AZURE_OPENAI_MODEL")
model_secondary = os.getenv("AZURE_OPENAI_MODEL_SECONDARY")

print("Primary model   :", model_primary)
print("Secondary model :", model_secondary)

Primary model   : gpt-4.1-mini
Secondary model : gpt-5-mini


In [38]:
comparison_prompt = """
A member has completed 10 physical therapy visits.

Policy:
Prior authorization is required starting from the 11th visit.

Current request:
The provider is requesting the member's 11th physical therapy visit.

Answer using exactly this format:

Decision:
Reason:
"""

In [39]:
primary_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": comparison_prompt
        }
    ]
)

secondary_response = client.chat.completions.create(
    model=model_secondary,
    messages=[
        {
            "role": "user",
            "content": comparison_prompt
        }
    ]
)

print("PRIMARY MODEL:", model_primary)
print("-" * 60)
print(primary_response.choices[0].message.content)

print("\nSECONDARY MODEL:", model_secondary)
print("-" * 60)
print(secondary_response.choices[0].message.content)

PRIMARY MODEL: gpt-4.1-mini
------------------------------------------------------------
Decision: Approved  
Reason: The current request is for the member's 11th physical therapy visit, and prior authorization is required starting from the 11th visit. Since this is the first visit requiring authorization, the request is approved.

SECONDARY MODEL: gpt-5-mini
------------------------------------------------------------
Decision: Denied — prior authorization is required for the 11th physical therapy visit and no approved authorization is on file.
Reason: Policy requires prior authorization starting with visit 11; the member has completed 10 visits and the current request for visit 11 lacks an approved prior authorization.


In [40]:
comparison_prompt_guarded = """
A member has completed 10 physical therapy visits.

Policy:
Prior authorization is required starting from the 11th visit.

Current request:
The provider is requesting the member's 11th physical therapy visit.

Rules:
1. Use only the information provided.
2. Do not assume whether authorization has already been requested, approved, or denied.
3. If the information is insufficient for an approval/denial decision, state that clearly.

Answer using exactly this format:

Prior Authorization Required:
Approval Decision:
Reason:
"""

In [41]:
primary_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": comparison_prompt
        }
    ]
)

secondary_response = client.chat.completions.create(
    model=model_secondary,
    messages=[
        {
            "role": "user",
            "content": comparison_prompt
        }
    ]
)

print("PRIMARY MODEL:", model_primary)
print("-" * 60)
print(primary_response.choices[0].message.content)

print("\nSECONDARY MODEL:", model_secondary)
print("-" * 60)
print(secondary_response.choices[0].message.content)

PRIMARY MODEL: gpt-4.1-mini
------------------------------------------------------------
Decision: Approved  
Reason: The member has completed 10 physical therapy visits, and prior authorization is required starting from the 11th visit. Since the current request is for the 11th visit and prior authorization is being requested, the request can be approved.

SECONDARY MODEL: gpt-5-mini
------------------------------------------------------------
Decision: Prior authorization required.
Reason: The member has completed 10 PT visits and policy mandates prior authorization beginning with the 11th visit; the current request is for the 11th visit and therefore requires prior authorization before it can be authorized.


In [42]:
def show_metrics(name, response):
    print(name)
    print("-" * 50)

    print("Actual model       :", response.model)
    print("Prompt tokens      :", response.usage.prompt_tokens)
    print("Completion tokens  :", response.usage.completion_tokens)
    print("Total tokens       :", response.usage.total_tokens)

    latency = getattr(response.usage, "latency_checkpoint", None)

    if latency:
        print("Total latency (ms) :", latency.get("total_duration_ms"))
        print("First token (ms)   :", latency.get("user_visible_ttft_ms"))

    print()


show_metrics("GPT-4.1-mini", primary_response)
show_metrics("GPT-5-mini", secondary_response)

GPT-4.1-mini
--------------------------------------------------
Actual model       : gpt-4.1-mini-2025-04-14
Prompt tokens      : 59
Completion tokens  : 54
Total tokens       : 113
Total latency (ms) : 1390
First token (ms)   : 464

GPT-5-mini
--------------------------------------------------
Actual model       : gpt-5-mini-2025-08-07
Prompt tokens      : 58
Completion tokens  : 700
Total tokens       : 758
Total latency (ms) : 9880
First token (ms)   : 413



In [43]:
print("GPT-4.1-mini token details")
print(primary_response.usage.completion_tokens_details)

print("\nGPT-5-mini token details")
print(secondary_response.usage.completion_tokens_details)

GPT-4.1-mini token details
CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0)

GPT-5-mini token details
CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=640, rejected_prediction_tokens=0)


In [44]:
model_comparison = pd.DataFrame([
    {
        "Model": "GPT-4.1-mini",
        "Prompt Tokens": primary_response.usage.prompt_tokens,
        "Completion Tokens": primary_response.usage.completion_tokens,
        "Reasoning Tokens": primary_response.usage.completion_tokens_details.reasoning_tokens,
        "Total Tokens": primary_response.usage.total_tokens,
        "Latency (ms)": primary_response.usage.latency_checkpoint["total_duration_ms"],
        "Observed Behavior": "Fast, but made unsupported approval inference"
    },
    {
        "Model": "GPT-5-mini",
        "Prompt Tokens": secondary_response.usage.prompt_tokens,
        "Completion Tokens": secondary_response.usage.completion_tokens,
        "Reasoning Tokens": secondary_response.usage.completion_tokens_details.reasoning_tokens,
        "Total Tokens": secondary_response.usage.total_tokens,
        "Latency (ms)": secondary_response.usage.latency_checkpoint["total_duration_ms"],
        "Observed Behavior": "Better evidence discipline, but higher reasoning cost"
    }
])

model_comparison

,Model,Prompt Tokens,Completion Tokens,Reasoning Tokens,Total Tokens,Latency (ms),Observed Behavior
0,GPT-4.1-mini,59,54,0,113,1390,"Fast, but made unsupported approval inference"
1,GPT-5-mini,58,700,640,758,9880,"Better evidence discipline, but higher reasoni..."


In [45]:
weak_prompt = """
Explain prior authorization.
"""

weak_prompt_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": weak_prompt
        }
    ]
)

print(weak_prompt_response.choices[0].message.content)
print("\nTotal tokens:", weak_prompt_response.usage.total_tokens)

Prior authorization is a process used by health insurance companies to determine if they will cover a prescribed medication, treatment, or medical procedure. Before the service is provided, the healthcare provider must obtain approval from the insurer to ensure that the requested service is medically necessary and meets the insurer's coverage criteria.

The process typically involves the healthcare provider submitting a request that includes clinical information about the patient's condition and the rationale for the treatment. The insurance company then reviews this information and decides whether to approve or deny coverage.

The purpose of prior authorization is to control costs, prevent unnecessary or inappropriate treatments, and ensure patient safety. However, it can sometimes delay access to care if the authorization takes time to process.

Total tokens: 150


In [46]:
strong_prompt = """
You are a healthcare payer domain assistant.

Explain prior authorization to a healthcare technology professional.

Requirements:
1. Use payer terminology.
2. Explain the purpose and workflow.
3. Keep the answer to exactly 3 bullet points.
4. Maximum 80 words.
5. Do not add information beyond the requested scope.
"""

strong_prompt_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": strong_prompt
        }
    ]
)

print(strong_prompt_response.choices[0].message.content)
print("\nTotal tokens:", strong_prompt_response.usage.total_tokens)

- Prior authorization (PA) is a utilization management process where payers require approval before covering specific services or medications to ensure medical necessity and cost-effectiveness.  
- Providers submit PA requests via electronic portals or fax, including clinical documentation; payers review against evidence-based guidelines and benefit policies.  
- Approved PAs enable claims adjudication with reimbursement; denials may lead to appeals or alternative treatments, optimizing care quality and controlling spend.

Total tokens: 159


In [47]:
print("WEAK PROMPT")
print("Prompt tokens     :", weak_prompt_response.usage.prompt_tokens)
print("Completion tokens :", weak_prompt_response.usage.completion_tokens)
print("Total tokens      :", weak_prompt_response.usage.total_tokens)

print("\nSTRONG PROMPT")
print("Prompt tokens     :", strong_prompt_response.usage.prompt_tokens)
print("Completion tokens :", strong_prompt_response.usage.completion_tokens)
print("Total tokens      :", strong_prompt_response.usage.total_tokens)

WEAK PROMPT
Prompt tokens     : 12
Completion tokens : 138
Total tokens      : 150

STRONG PROMPT
Prompt tokens     : 71
Completion tokens : 88
Total tokens      : 159


In [48]:
prompt_comparison = pd.DataFrame([
    {
        "Prompt Type": "Weak",
        "Prompt Tokens": weak_prompt_response.usage.prompt_tokens,
        "Completion Tokens": weak_prompt_response.usage.completion_tokens,
        "Total Tokens": weak_prompt_response.usage.total_tokens,
        "Control Level": "Low",
        "Output Structure": "Uncontrolled"
    },
    {
        "Prompt Type": "Strong",
        "Prompt Tokens": strong_prompt_response.usage.prompt_tokens,
        "Completion Tokens": strong_prompt_response.usage.completion_tokens,
        "Total Tokens": strong_prompt_response.usage.total_tokens,
        "Control Level": "High",
        "Output Structure": "Controlled"
    }
])

prompt_comparison


,Prompt Type,Prompt Tokens,Completion Tokens,Total Tokens,Control Level,Output Structure
0,Weak,12,138,150,Low,Uncontrolled
1,Strong,71,88,159,High,Controlled
